In [1]:
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Function
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    classification_report, balanced_accuracy_score,
    roc_auc_score, f1_score, confusion_matrix,
    matthews_corrcoef, cohen_kappa_score
)
import seaborn as sns
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import numpy as np

In [2]:
with open('/home/matildxi/Secretária/Documentos/Tese/DANN/wesad/processed_data_sequences.pkl', 'rb') as f:
    data = pickle.load(f)

X_seq      = data['X_seq']
y_seq      = data['y_seq']
groups_seq = data['groups_seq']

le = LabelEncoder()
domain_labels = le.fit_transform(groups_seq)

In [3]:
print(X_seq.shape)
print(y_seq.shape)
print(np.unique(y_seq))

(27543, 16, 60)
(27543,)
[0 1]


In [4]:
class CNNGRUHybrid(nn.Module):
    def __init__(self, in_channels=16, num_classes=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        # output da CNN: (batch, 128, 60) -> GRU espera (batch, 60, 128)
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.2,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)              # (batch, 128, 60)
        x = x.permute(0, 2, 1)      # (batch, 60, 128)
        out, _ = self.gru(x)
        out = out[:, -1, :]          # último timestep
        return self.classifier(out)

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logo   = LeaveOneGroupOut()

all_preds  = []
all_labels = []
fold_f1s   = []

subjects = np.unique(groups_seq)
print(f"Total subjects: {len(subjects)}\n")

for fold, (train_idx, test_idx) in enumerate(logo.split(X_seq, y_seq, groups=groups_seq)):
    subject_id = groups_seq[test_idx[0]]
    print(f"Fold {fold+1:2d} | Test subject: {subject_id}")

    X_train_raw = X_seq[train_idx]   # (n, 16, 60)
    y_train_raw = y_seq[train_idx]
    X_test_raw  = X_seq[test_idx]
    y_test_raw  = y_seq[test_idx]

    # SMOTE precisa de input 2D → flatten, resample, reshape de volta
    n_train, C, T = X_train_raw.shape
    X_flat = X_train_raw.reshape(n_train, C * T)

    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_flat, y_train_raw)
    X_res = X_res.reshape(-1, C, T)

    # Tensores
    X_tr = torch.tensor(X_res,       dtype=torch.float32)
    y_tr = torch.tensor(y_res,       dtype=torch.long)
    X_te = torch.tensor(X_test_raw,  dtype=torch.float32)
    y_te = torch.tensor(y_test_raw,  dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=64)

    # Modelo e otimizador frescos por fold
    model     = CNNGRUHybrid().to(device)
    criterion = nn.CrossEntropyLoss()          # SMOTE já balanceou, sem pesos
    optimizer = optim.Adam(model.parameters(), lr=5e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    # Treino
    EPOCHS = 50
    best_f1 = 0
    patience = 10
    no_improve = 0
    
    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        scheduler.step()
    
        # validação rápida no próprio test fold (só para early stopping)
        model.eval()
        ep_preds = []
        with torch.no_grad():
            for xb, _ in test_loader:
                ep_preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        ep_f1 = f1_score(y_te.numpy(), ep_preds, average='macro')
    
        if ep_f1 > best_f1:
            best_f1 = ep_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"           Early stop no epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)

    # Avaliação
    model.eval()
    fold_preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            fold_preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())

    f1 = f1_score(y_te.numpy(), fold_preds, average='macro')
    fold_f1s.append(f1)
    all_preds.extend(fold_preds)
    all_labels.extend(y_te.numpy())
    print(f"           Macro F1: {f1:.4f}\n")

Total subjects: 15

Fold  1 | Test subject: S10
           Early stop no epoch 30
           Macro F1: 0.9884

Fold  2 | Test subject: S11
           Early stop no epoch 15
           Macro F1: 0.8074

Fold  3 | Test subject: S13
           Early stop no epoch 11
           Macro F1: 0.7907

Fold  4 | Test subject: S14
           Early stop no epoch 11
           Macro F1: 0.9727

Fold  5 | Test subject: S15
           Early stop no epoch 15
           Macro F1: 0.8969

Fold  6 | Test subject: S16
           Early stop no epoch 12
           Macro F1: 0.9953

Fold  7 | Test subject: S17
           Early stop no epoch 12
           Macro F1: 0.9661

Fold  8 | Test subject: S2
           Early stop no epoch 15
           Macro F1: 0.9252

Fold  9 | Test subject: S3
           Early stop no epoch 28
           Macro F1: 0.8305

Fold 10 | Test subject: S4
           Early stop no epoch 17
           Macro F1: 0.9693

Fold 11 | Test subject: S5
           Early stop no epoch 11
           M

In [6]:
print("=" * 50)
print(f"Macro F1 por fold: {[f'{v:.4f}' for v in fold_f1s]}")
print(f"Média: {np.mean(fold_f1s):.4f}  |  Std: {np.std(fold_f1s):.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=['no-stress', 'stress']))

Macro F1 por fold: ['0.9884', '0.8074', '0.7907', '0.9727', '0.8969', '0.9953', '0.9661', '0.9252', '0.8305', '0.9693', '0.9928', '0.8990', '0.9497', '0.9688', '0.7097']
Média: 0.9108  |  Std: 0.0847

              precision    recall  f1-score   support

   no-stress       0.95      0.92      0.93     17577
      stress       0.86      0.92      0.89      9966

    accuracy                           0.92     27543
   macro avg       0.91      0.92      0.91     27543
weighted avg       0.92      0.92      0.92     27543



In [ ]:
# F1 @ Youden-optimal threshold
from sklearn.metrics import roc_curve, f1_score
import numpy as np
_yt = np.array(y_test); _ys = np.array(y_score)
_fpr, _tpr, _thr = roc_curve(_yt, _ys)
_t_opt = float(_thr[np.argmax(_tpr - _fpr)])
_f1opt = f1_score(_yt, (_ys >= _t_opt).astype(int), average='macro', zero_division=0)
print(f'F1@opt : {_f1opt:.3f}  (threshold={_t_opt:.3f})')